# 10 · Multi-Agent and Subgraph Patterns

One agent + 25 tools = wrong tool picks, bloated prompts, polluted context. The fix mirrors human teams: **specialists with narrow toolsets, a supervisor to coordinate**.

```
 user ──► SUPERVISOR ──► answer
             │ delegates (as tool calls)
      ┌──────┴──────┐
      ▼             ▼
 research agent   math agent
```

The trick that makes it almost free: **an agent is invocable → an agent can be a tool.**

---

## Setup

In [ ]:
%pip install -qU langchain langchain-openai python-dotenv

In [ ]:
import os
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain_core.tools import tool
from dotenv import load_dotenv

load_dotenv("../../.env")

llm = init_chat_model(os.environ["CHAT_MODEL"], temperature=0)

---
## 1. Specialist 1: a research agent

Narrow toolset (one fake internal-docs search), its own tuned system prompt. NovaPlay is fictional — the model can't cheat.

In [ ]:
KB = {
    "users": "NovaPlay has 2.4 million monthly active users (Q2 2026).",
    "arpu": "NovaPlay's average revenue per user (ARPU) is $3.20 per month.",
    "churn": "NovaPlay's monthly churn rate is 2.1%.",
    "founded": "NovaPlay was founded in 2021 in Berlin.",
}

@tool
def search_docs(query: str) -> str:
    """Search NovaPlay's internal docs. Use keywords like 'users', 'arpu', 'churn', 'founded'."""
    q = query.lower()
    hits = [fact for key, fact in KB.items() if key in q]
    return "\n".join(hits) or "No results. Available topics: " + ", ".join(KB)

research_agent = create_agent(
    llm, [search_docs],
    system_prompt="You are a research specialist for NovaPlay. Answer ONLY from search_docs "
                  "results — never guess. Reply with the facts, concisely.",
)

## 2. Specialist 2: a math agent

In [ ]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b

math_agent = create_agent(
    llm, [multiply, divide],
    system_prompt="You are a calculator. Always use tools for arithmetic; show the result clearly.",
)

---
## 3. Wrap each specialist as a tool

The docstring is the **delegation contract** — it tells the supervisor *when* to hand off. The specialist's internal tool trace stays private; only the distilled answer crosses the boundary.

In [ ]:
@tool
def research(question: str) -> str:
    """Look up facts about NovaPlay (users, revenue, churn, history). Ask ONE specific question."""
    out = research_agent.invoke({"messages": [("human", question)]})
    return out["messages"][-1].content

@tool
def calculate(problem: str) -> str:
    """Solve an arithmetic problem stated in plain words, e.g. 'multiply 2.4 million by 3.20'."""
    out = math_agent.invoke({"messages": [("human", problem)]})
    return out["messages"][-1].content

## 4. The supervisor

...is just an ordinary `create_agent` whose tools happen to be agents.

In [ ]:
supervisor = create_agent(
    llm, [research, calculate],
    system_prompt="You coordinate specialists. Delegate lookups to `research` and arithmetic "
                  "to `calculate` — never answer domain questions or do math yourself.",
)

out = supervisor.invoke({
    "messages": [("human",
        "Estimate NovaPlay's monthly revenue, assuming every active user pays the ARPU.")]
})
print(out["messages"][-1].content)

### Watch the delegation live

Stream the supervisor's loop (lesson 06's trick): it researches first, then sends the numbers to the math specialist.

In [ ]:
for step in supervisor.stream(
    {"messages": [("human",
        "How many users does NovaPlay lose to churn each month? Show the calculation.")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

> Two graphs just ran *inside* a third — the supervisor never saw their internal traces, only distilled answers. Clean context by construction.

---
## 5. Under the hood: subgraphs

`create_agent` returns a **compiled LangGraph graph** — so agent-as-tool is really *graph inside graph*. LangGraph makes this first-class: **a compiled graph is a valid node**. Here are the bare mechanics (no LLM, so the wiring is obvious):

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str

# Child graph: two-step text normalizer.
child_builder = StateGraph(State)
child_builder.add_node("trim", lambda s: {"text": s["text"].strip()})
child_builder.add_node("shout", lambda s: {"text": s["text"].upper()})
child_builder.add_edge(START, "trim")
child_builder.add_edge("trim", "shout")
child_builder.add_edge("shout", END)
normalizer = child_builder.compile()

# Parent graph: uses the ENTIRE child graph as one node.
parent_builder = StateGraph(State)
parent_builder.add_node("normalize", normalizer)          # ← compiled graph as node
parent_builder.add_node("exclaim", lambda s: {"text": s["text"] + "!!!"})
parent_builder.add_edge(START, "normalize")
parent_builder.add_edge("normalize", "exclaim")
parent_builder.add_edge("exclaim", END)
pipeline = parent_builder.compile()

print(pipeline.invoke({"text": "   hello subgraphs   "}))

Two ways to embed a graph in a graph — pick by who should control the flow:

| Embedding | Flow decided by | Use when |
|---|---|---|
| Agent-as-tool | the supervisor **LLM**, at runtime | which specialist is needed depends on the request |
| Graph-as-node | **you**, with fixed edges | the stage order is known and must be deterministic |

---
## What to remember

| Concept | What it does |
|---|---|
| Agents-as-tools | `@tool` wrapping `specialist.invoke(...)` → supervisor delegates naturally |
| Supervisor | An ordinary agent whose tools are agents |
| Docstring = delegation contract | Tells the supervisor when to hand off |
| Tool boundary = context boundary | Distilled task in, distilled answer out; traces stay private |
| `add_node("x", child.compile())` | A compiled graph is a valid node — subgraphs are first-class |
| Escalation ladder | chain → parallel → router → agent → multi-agent; stop at the first rung that works |

**Next (Phase: Production):** 11 — **Streaming and async**: make everything feel fast (tokens as they generate) and be fast (concurrent calls).